# Preprocessing

In [1]:
#Importing libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt

In [2]:
#Storing the movie information into a pandas dataframe
movies_df = pd.read_csv("Desktop\\movies.csv")
#Storing the user information into a pandas dataframe
ratings_df = pd.read_csv("Desktop\\ratings.csv")
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


#### Removing the year from the "title" column and storing in a new "year" column

In [3]:
#We specify the parantheses so we don't conflict with movies that have years in their titles(regex style)
movies_df["year"] = movies_df.title.str.extract("(\(\d\d\d\d\))",expand = False)
#Removing the parantheses
movies_df["year"] = movies_df.year.str.extract("(\d\d\d\d)",expand = False)
#Removing the years from the "title" column
movies_df["title"] = movies_df.title.str.replace("(\(\d\d\d\d\))", "", regex = True)
#Applying the strip function to get rid of any ending whitespace characters
movies_df["title"] = movies_df["title"].apply(lambda x: x.strip())
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,2,Jumanji,Adventure|Children|Fantasy,1995
2,3,Grumpier Old Men,Comedy|Romance,1995
3,4,Waiting to Exhale,Comedy|Drama|Romance,1995
4,5,Father of the Bride Part II,Comedy,1995


#### Splitting the values in the "genres" column into a list of genres to simplify for future use

In [4]:
movies_df["genres"] = movies_df.genres.str.split("|")
movies_df.head()

,movieId,title,genres,year
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995
2,3,Grumpier Old Men,"[Comedy, Romance]",1995
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995
4,5,Father of the Bride Part II,[Comedy],1995


In [5]:
#Copying the movies dataframe into a new one since we don't needto use the genre information
movies_with_genres_df = movies_df.copy()
#For every row in the dataframe, iterate through the list of genres and place a 1 into the cells
for index, row in movies_df.iterrows():
    for genre in row["genres"]:
        movies_with_genres_df.at[index, genre] = 1
#Filling in the NaN values with 0 to show that a movies doesn't have that column's genre
movies_with_genres_df = movies_with_genres_df.fillna(0)
movies_with_genres_df.head()

,movieId,title,genres,year,Adventure,Animation,Children,Comedy,Fantasy,Romance,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,Grumpier Old Men,"[Comedy, Romance]",1995,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,Waiting to Exhale,"[Comedy, Drama, Romance]",1995,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,Father of the Bride Part II,[Comedy],1995,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### ratings dataframe

In [6]:
ratings_df.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [7]:
#We don't need timestamp column
ratings_df = ratings_df.drop(columns = ["timestamp"])
ratings_df.head()

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0


# Content-Based Filtering

In [8]:
user_input = [
    {'title':'Breakfast Club', 'rating':5},
    {'title':'Toy Story', 'rating':3.5},
    {'title':'Jumanji', 'rating':2},
    {'title':'Pulp Fiction', 'rating':5},
    {'title':'Akira', 'rating':4.5}
]
input_movies = pd.DataFrame(user_input)
input_movies

,title,rating
0,Breakfast Club,5.0
1,Toy Story,3.5
2,Jumanji,2.0
3,Pulp Fiction,5.0
4,Akira,4.5


#### Add movieId to input user

In [9]:
#Filtering out the movies by title
input_id = movies_df[movies_df["title"].isin(input_movies["title"].tolist())]
#Then merging it so we can get the movieID
input_movies = pd.merge(input_id, input_movies)
#Dropping information we won't use
input_movies = input_movies.drop(columns = ["genres", "year"])
input_movies

,movieId,title,rating
0,1,Toy Story,3.5
1,2,Jumanji,2.0
2,296,Pulp Fiction,5.0
3,1274,Akira,4.5


In [10]:
user_movies = movies_with_genres_df[movies_with_genres_df["movieId"].isin(input_movies["movieId"].tolist())]
user_movies

,movieId,title,genres,year,Adventure,Animation,Children,Comedy,Fantasy,Romance,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1,Toy Story,"[Adventure, Animation, Children, Comedy, Fantasy]",1995,1.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Jumanji,"[Adventure, Children, Fantasy]",1995,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
257,296,Pulp Fiction,"[Comedy, Crime, Drama, Thriller]",1994,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
973,1274,Akira,"[Action, Adventure, Animation, Sci-Fi]",1988,1.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
#Reseting the index to avoid future issues
user_movies = user_movies.reset_index(drop = True)
#Dropping unnecessary issues
user_genre_table = user_movies.drop(columns = ["movieId", "title", "genres", "year"])
user_genre_table

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Learning the input's preferences: turning each genre into weights by using the input's reviews and multiplying them into the input's genre table and then summing up the resulting table by column

In [12]:
input_movies["rating"]

0    3.5
1    2.0
2    5.0
3    4.5
Name: rating, dtype: float64

In [13]:
#weights
user_profile  = user_genre_table.transpose().dot(input_movies["rating"])
user_profile

Adventure             10.0
Animation              8.0
Children               5.5
Comedy                 8.5
Fantasy                5.5
Romance                0.0
Drama                  5.0
Action                 4.5
Crime                  5.0
Thriller               5.0
Horror                 0.0
Mystery                0.0
Sci-Fi                 4.5
War                    0.0
Musical                0.0
Documentary            0.0
IMAX                   0.0
Western                0.0
Film-Noir              0.0
(no genres listed)     0.0
dtype: float64

#### Now we have the weights for every of the user's preferences

In [14]:
genre_table = movies_with_genres_df.set_index(movies_with_genres_df["movieId"])
genre_table = genre_table.drop(columns = ["movieId", "title", "genres", "year"])
genre_table.head()

,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,Action,Crime,Thriller,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
movieId,,,,,,,,,,,,,,,,,,,,
1,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
genre_table.shape

(9742, 20)

#### Taking the weighted average of every movie based on the input profile and recommending the top 20 movies

In [16]:
#Multiply the genres by the weights and then take the weighted average
recommendation_table_df = ((genre_table * user_profile).sum(axis = 1)) / (user_profile.sum())
recommendation_table_df.head()

movieId
1    0.609756
2    0.341463
3    0.138211
4    0.219512
5    0.138211
dtype: float64

In [17]:
#Sorting the recommendations in descending order
recommendation_table_df = recommendation_table_df.sort_values(ascending = False)
recommendation_table_df.head()

movieId
2987      0.691057
134853    0.691057
26340     0.682927
130520    0.682927
51939     0.682927
dtype: float64

In [18]:
#The final recommendation table!
movies_df.loc[movies_df["movieId"].isin(recommendation_table_df.head(20).keys())]

,movieId,title,genres,year
478,546,Super Mario Bros.,"[Action, Adventure, Children, Comedy, Fantasy,...",1993
559,673,Space Jam,"[Adventure, Animation, Children, Comedy, Fanta...",1996
2250,2987,Who Framed Roger Rabbit?,"[Adventure, Animation, Children, Comedy, Crime...",1988
4631,6902,Interstate 60,"[Adventure, Comedy, Drama, Fantasy, Mystery, S...",2002
5490,26340,"Twelve Tasks of Asterix, The (Les douze travau...","[Action, Adventure, Animation, Children, Comed...",1976
5620,27155,"Batman/Superman Movie, The","[Action, Adventure, Animation, Children, Fanta...",1998
5819,32031,Robots,"[Adventure, Animation, Children, Comedy, Fanta...",2005
6047,40339,Chicken Little,"[Action, Adventure, Animation, Children, Comed...",2005
6448,51939,TMNT (Teenage Mutant Ninja Turtles),"[Action, Adventure, Animation, Children, Comed...",2007
6455,52287,Meet the Robinsons,"[Action, Adventure, Animation, Children, Comed...",2007
